# 07 — Deep Learning Results

Architecture comparisons, R² metrics and validation curves. Biological figures are generated by notebook 10.


In [ ]:
from pathlib import Path
import os

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.resolve()
    for candidate in [start] + list(start.parents):
        if (candidate / "config" / "genes.json").exists():
            return candidate
    raise FileNotFoundError("Could not find project root. Start Jupyter from repo root or notebooks folder.")

PROJECT_ROOT = find_project_root()
shared_notebook = PROJECT_ROOT / "notebooks" / "_Shared_Training_Utilities.ipynb"
get_ipython().run_line_magic("run", str(shared_notebook))

In [ ]:
metrics = aggregate_all_metrics()
histories = aggregate_all_histories()
summary = summarize_experiments(metrics)

print("Metrics shape:", metrics.shape)
print("Histories shape:", histories.shape)
display(summary)

In [ ]:
# Per-target metric table
if not metrics.empty:
    per_target = metrics.sort_values(["target", "r2"], ascending=[True, False])
    display(per_target.head(20))
else:
    per_target = pd.DataFrame()

In [ ]:
# Best experiment by target
if not metrics.empty:
    idx = metrics.groupby("target")["r2"].idxmax()
    best_by_target = metrics.loc[idx].sort_values("target")
    display(best_by_target[["target", "experiment_name", "model_family", "r2", "mae", "rmse"]])
else:
    best_by_target = pd.DataFrame()

## Visual: mean R² by experiment

In [ ]:
if not summary.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    plot_df = summary.sort_values("mean_r2", ascending=True)
    ax.barh(plot_df["experiment_name"], plot_df["mean_r2"])
    ax.set_title("Mean R² by experiment")
    ax.set_xlabel("Mean R² across TAA targets")
    plt.tight_layout()
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURES_DIR / "mean_r2_by_experiment.png", dpi=300, bbox_inches="tight")
    plt.show()

## Visual: per-target R² by experiment

In [ ]:
if not metrics.empty:
    r2_pivot = metrics.pivot_table(index="target", columns="experiment_name", values="r2", aggfunc="mean").sort_index()

    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(r2_pivot.index))
    width = 0.8 / max(len(r2_pivot.columns), 1)

    for i, experiment_name in enumerate(r2_pivot.columns):
        ax.bar(x + i * width, r2_pivot[experiment_name], width=width, label=experiment_name)

    ax.set_xticks(x + width * (len(r2_pivot.columns) - 1) / 2)
    ax.set_xticklabels(r2_pivot.index, rotation=45, ha="right")
    ax.set_ylabel("R²")
    ax.set_title("Per-target R² by experiment")
    ax.axhline(0, linewidth=1)
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "per_target_r2_by_experiment.png", dpi=300, bbox_inches="tight")
    plt.show()

## Visual: training curves